# Yelp Review Data Cleaning Script

This script reads Yelp review datasets (by state) from CSV files, performs text preprocessing, and saves cleaned versions for analysis.

## Features
- Loads large CSV files in chunks to manage memory efficiently.
- Selects only relevant columns: `review_stars` and `review_text`.
- Cleans review text by:
  - Lowercasing
  - Handling negations
  - Removing URLs, numbers, punctuation, and stopwords
- Generates additional features:
  - `review_length` — number of words in the review
  - `num_exclamations` — number of exclamation marks
  - `num_caps_words` — number of fully capitalized words
  - `clean_text` — cleaned version of the review text
- Saves each cleaned dataset as a CSV in the `cleaned-data/` folder.


In [1]:
import glob
import nltk
from nltk.corpus import stopwords # might have to run nltk.download('stopwords')
import numpy as np
import os
import pandas as pd
import re
import string

In [3]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /Users/cole/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [4]:
folder = "../Data-Cleaning/associated-data-by-state/"
csv_files = glob.glob(os.path.join(folder, "*.csv"))
dfs = {}
for file in csv_files:
    name = os.path.splitext(os.path.basename(file))[0].split("_")[-1]
    chunks = pd.read_csv(file, chunksize=100000, low_memory=False, usecols=['review_stars', 'review_text'])
    dfs[name] = pd.concat(chunks, ignore_index=True)

In [5]:
def clean_text(text):
    # Lowercase
    text = text.lower()
    # Handling negations
    text = re.sub(r"not\s+(\w+)", r"not_\1", text)
    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)
    # Remove numbers
    text = re.sub(r"\d+", "", text)
    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))
    # Remove stopwords
    text = " ".join([word for word in text.split() if word not in stop_words])
    return text

In [6]:
for df in dfs:
    dfs[df].rename(columns={"review_text": "text", "review_stars": "stars"}, inplace=True)
    dfs[df]["review_length"] = dfs[df]["text"].apply(lambda x: len(x.split()))
    dfs[df]["num_exclamations"] = dfs[df]["text"].str.count("!")
    dfs[df]["num_caps_words"] = dfs[df]["text"].apply(lambda x: sum(1 for w in x.split() if w.isupper()))
    dfs[df]["clean_text"] = dfs[df]["text"].astype(str).apply(clean_text)

In [7]:
# Saving all of these datasets in csv format
folder_name = 'cleaned-data'
if not os.path.exists(folder_name):
    os.makedirs(folder_name)
    
for df in dfs:
    filename = f"{folder_name}/{df}.csv"
    dfs[df].to_csv(filename, index=False)
    print("Saved: ", filename)

Saved:  cleaned-data/DE.csv
Saved:  cleaned-data/FL.csv
Saved:  cleaned-data/PA.csv
Saved:  cleaned-data/ID.csv
Saved:  cleaned-data/CA.csv
Saved:  cleaned-data/MO.csv
Saved:  cleaned-data/NC.csv
Saved:  cleaned-data/AZ.csv
Saved:  cleaned-data/TN.csv
Saved:  cleaned-data/XMS.csv
Saved:  cleaned-data/NV.csv
Saved:  cleaned-data/AB.csv
Saved:  cleaned-data/MT.csv
Saved:  cleaned-data/NJ.csv
Saved:  cleaned-data/LA.csv
Saved:  cleaned-data/CO.csv
Saved:  cleaned-data/HI.csv
Saved:  cleaned-data/IL.csv
Saved:  cleaned-data/IN.csv
